In [1]:
import numpy as np
import pandas as pd
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
import re
import ftfy
import html
pd.set_option('display.max_colwidth', None)

In [2]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [3]:
#TODO:cambiare le print in inglese, in futuro dovrò fare catboost/target encoding
n_nan = df['source'].isna().sum()
n_placeholder = (df['source'] == '\\N').sum()
n_empty = (df['source'].astype(str).str.strip() == '').sum()

print(f"NaN: {n_nan}")
print(f"Placeholder \\N: {n_placeholder}")
print(f"Stringhe vuote: {n_empty}")

df['source'] = df['source'].replace('\\N', np.nan)
df['source'] = df['source'].replace('', np.nan)
df['source'] = df['source'].fillna('Other')

min_freq = 5  
source_counts = df['source'].value_counts()
sources_kept = source_counts[source_counts >= min_freq].index.tolist()

df['source'] = np.where(df['source'].isin(sources_kept),df['source'],'Other')

final_source_counts = df['source'].value_counts()

print(f"Numero categorie finali (incluso Other): {len(final_source_counts)}")

NaN: 0
Placeholder \N: 294
Stringhe vuote: 0
Numero categorie finali (incluso Other): 489


### *Title* feature inspection

In [4]:
n_nan_title = df['title'].isna().sum()
n_placeholders_title = (df['title']== '\\N').sum()
n_empty_title = (df['title'].astype(str).str.strip()=='').sum()

print(f"Number of NaN rows: {n_nan_title}")
print(f"Number of placeholders (\\N): {n_placeholders_title}")
print(f"Number of empty rows: {n_empty_title}")
print("Titles Sample:")
print(df['title'].sample(10))

Number of NaN rows: 1
Number of placeholders (\N): 0
Number of empty rows: 2
Titles Sample:
Id
25395                       Americans a Bit Taller, Much Heavier, Report Says
60505                                                       Hu&#39;s on First
69694                 McCain plays up reformer image in TV ads \\n    (AP)\\n
16468                                    Anti-smoking ads to be toughest ever
63029                          O.J. Simpson leaves jail after judge's lecture
25752                                                         B-R comes alive
50511        Ukraine parliament approves Tymoshenko as PM \\n    (Reuters)\\n
61127                                       'Million suffer' in Africa floods
865      News Corp in talks to take MySpace to China: WSJ \\n    (Reuters)\\n
31328                             ADV: Try Currency Trading Risk-Free 30 Days
Name: title, dtype: object


### *Article* feature inspection

In [5]:
n_nan_article = df['article'].isna().sum()
n_placeholders_article = (df['article']=='\\N').sum()
n_empty_article = (df['article'].astype(str).str.strip()=='').sum()

print(f"Number of NaN rows: {n_nan_article}")
print(f"Number of placeholders (\\N): {n_placeholders_article}")
print(f"Number of empty rows: {n_empty_article}")
print("Articles Sample")
print(df['article'].sample(10))

Number of NaN rows: 1
Number of placeholders (\N): 1874
Number of empty rows: 7
Articles Sample
Id
55773                                                                                                                                                                                                                                                                  Afghanistan has made sufficient steps to improve its economy to qualify for debt relief.
58076     PARIS/LONDON (Reuters) - Airbus parent firm EADS <A HREF="http://www.investor.reuters.com/FullQuote.aspx?ticker=EAD.PA&target=/stocks/quickinfo/fullquote">EAD.PA</A>   will complete its review of the challenges it faces within   weeks, Co-Chief Executive Louis Gallois said on Thursday, while   dismissing negative comments from BAE Systems.
70950                                                                                                                                                             AP - There's an old saying, "Justic

### *PageRank* feature inspection

In [6]:
n_nan_pr = df['page_rank'].isna().sum()
n_placeholders_pr = (df['page_rank']=='\\N').sum()
n_empty_pr = (df['page_rank'].astype(str).str.strip()=='').sum()
rank_5=np.array([df['page_rank']==5]).sum()

print(f"Number of NaN rows: {n_nan_pr}")
print(f"Number of placeholders (\\N): {n_placeholders_pr}")
print(f"Number of empty rows: {n_empty_pr}")
print(f"Number of articles with PageRank 5: {rank_5}")

Number of NaN rows: 0
Number of placeholders (\N): 0
Number of empty rows: 0
Number of articles with PageRank 5: 73891


### *Timestamp* feature inspection 

In [7]:
n_nan_time = df['timestamp'].isna().sum()
n_placeholders_time = (df['timestamp']=='\\N').sum()
n_empty_time = (df['timestamp'].astype(str).str.strip()=='').sum()
n_uslesess_time=np.array([df['timestamp']=="0000-00-00 00:00:00"]).sum()

print(f"Number of NaN rows: {n_nan_time}")
print(f"Number of placeholders (\\N): {n_placeholders_time}")
print(f"Number of empty rows: {n_empty_time}")
print(f"Number invalid dates (0000-00-00 00:00:00): {n_uslesess_time}")
print(df['timestamp'].sample(10))

Number of NaN rows: 0
Number of placeholders (\N): 0
Number of empty rows: 0
Number invalid dates (0000-00-00 00:00:00): 27750
Id
28079    0000-00-00 00:00:00
39669    2004-09-16 10:51:57
3722     2007-02-13 16:16:30
53273    2007-03-05 12:41:30
37873    2007-12-25 04:11:30
50857    0000-00-00 00:00:00
66669    0000-00-00 00:00:00
12302    2007-06-07 23:02:23
11475    2007-05-29 08:07:17
3587     0000-00-00 00:00:00
Name: timestamp, dtype: object


### *Timestamp* feature processing

In [8]:
def process_timestamp(df):
    df = df.copy()

    dt = pd.to_datetime(df['timestamp'], errors='coerce')

    df['has_date'] = dt.notna().astype(int)
    df['quarter'] = dt.dt.quarter.fillna(-1).astype(int)
    df['is_weekend'] = dt.dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    df = df.drop(columns=['timestamp'])
    return df

df = process_timestamp(df)

timestamp_cols = ['has_date', 'is_weekend','quarter']
print(f"Timestamp expanded columns: {timestamp_cols}")
print("'timestamp' raw column deleted")
print("Sample of 10 timestamps:")
print(df[timestamp_cols].sample(10))

Timestamp expanded columns: ['has_date', 'is_weekend', 'quarter']
'timestamp' raw column deleted
Sample of 10 timestamps:
       has_date  is_weekend  quarter
Id                                  
61209         0           0       -1
41995         1           0        2
72341         1           0        1
4680          0           0       -1
6946          1           0        1
59017         1           0        4
79720         1           0        4
65585         1           0        1
11616         1           0        3
70505         1           0        1


### *Title* feature stemming

In [9]:
import re
import html
import ftfy
import pandas as pd

def clean_text_light(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = ftfy.fix_text(text)

    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def process_title_final(df):
    df = df.copy()

    df['title_clean'] = df['title'].apply(clean_text_light)
    df = df.drop(columns=['title'])

    return df


df = process_title_final(df)

print("Sample of 10 cleaned titles:")
print(df['title_clean'].sample(10).to_string(index=False))


Sample of 10 cleaned titles:
Id
                   gm u.s. sales down 24 percent in june
               hagel says us must address sudan genocide
                   air strikes target insurgents in iraq
                        officer killed in armed standoff
                chinese mark on africa means commerce ap
                       nato hails afghan mission success
                  dollar lower on weaker producer prices
federal deposit insurance corporation extends moratorium
               clement and boston near a three-year deal
                               protest song will play on


### *Article* feature stemming

In [ ]:
import re
import html
import ftfy
import pandas as pd

def clean_text_light(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Fix broken encodings
    text = ftfy.fix_text(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove common web/markup leftovers (keep it simple)
    text = re.sub(r'\b(a\s+href|href|img\s+src|nbsp|read\s+more|click\s+here)\b',' ',text,flags=re.IGNORECASE)


    # Lowercase
    text = text.lower()

    # Keep letters, numbers, and useful punctuation for n-grams
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def process_article_final(df):
    df = df.copy()

    # Normalize placeholders / missing
    df['article'] = df['article'].replace('\\N', '').fillna('').astype(str)

    df['article_clean'] = df['article'].apply(clean_text_light)
    df = df.drop(columns=['article'])

    return df


# === APPLY ===
df = process_article_final(df)

# === DEBUG / SANITY CHECK ===
print("Sample of 10 cleaned articles:")
print(df['article_clean'].sample(10).to_string(index=False))

Sample of 10 cleaned articles:
Id
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  new york reuters - blue-chip stocks fell on thursday as bond yields crept higher, stirring concern about rising interest rates as an increasing aversion to risk kept buyers at bay.
                                                                                                                                                                                                                                                                               

In [12]:
df['text'] = (df['title_clean'].fillna('') + ' ' + df['article_clean'].fillna('')).str.strip()

# opzionale: se vuoi ridurre memoria dopo aver creato text
df = df.drop(columns=['title_clean', 'article_clean'])


### Encoding categorical features

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from category_encoders.cat_boost import CatBoostEncoder

# ----------------------------
# Columns
# ----------------------------
TEXT_COL = 'text'
CAT_COLS = ['source']
NUM_COLS = ['page_rank', 'has_date', 'is_weekend', 'quarter']
TARGET_COL = 'label'

X = df[[TEXT_COL] + CAT_COLS + NUM_COLS]
y = df[TARGET_COL]

# ----------------------------
# Train / validation split
# ----------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ----------------------------
# Preprocessing
# ----------------------------
preprocess = ColumnTransformer(
    transformers=[
        (
            'tfidf',
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                max_features=200_000
            ),
            TEXT_COL
        ),
        (
            'catboost',
            CatBoostEncoder(
                cols=CAT_COLS,
                a=1.0,
                random_state=42
            ),
            CAT_COLS
        ),
        (
            'num',
            'passthrough',
            NUM_COLS
        )
    ]
)

# ----------------------------
# Fit preprocessing only (sanity check)
# ----------------------------
X_train_trans = preprocess.fit_transform(X_train, y_train)
X_val_trans = preprocess.transform(X_val)

print("Train shape:", X_train_trans.shape)
print("Validation shape:", X_val_trans.shape)


Train shape: (63997, 200005)
Validation shape: (16000, 200005)


In [16]:
import lightgbm as lgb
from sklearn.metrics import f1_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ----------------------------
# Class weights (important!)
# ----------------------------
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, class_weights))


lgb_clf = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(classes),
    class_weight=class_weight_dict,
    n_estimators=200,          # ↓↓↓
    learning_rate=0.1,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.6,     # ↓↓↓ CRITICO
    random_state=42,
    n_jobs=-1,
    verbose=-1                # toglie spam
)

lgb_clf.fit(X_train_trans, y_train)

y_val_pred = lgb_clf.predict(X_val_trans)
print("Validation Macro F1:",f1_score(y_val, y_val_pred, average='macro'))


/Users/giorgiozoccatelli/miniforge3/envs/data/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Validation Macro F1: 0.6844246898207639


In [17]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)

# Label mapping (dal pdf)
label_names = {
    0: "International News",
    1: "Business",
    2: "Technology",
    3: "Entertainment",
    4: "Sports",
    5: "General News",
    6: "Health"
}

# ---- 1) Metriche globali
macro_f1 = f1_score(y_val, y_val_pred, average='macro')
micro_f1 = f1_score(y_val, y_val_pred, average='micro')
weighted_f1 = f1_score(y_val, y_val_pred, average='weighted')

print("=== GLOBAL METRICS ===")
print(f"Macro F1    : {macro_f1:.6f}")
print(f"Micro F1    : {micro_f1:.6f}")
print(f"Weighted F1 : {weighted_f1:.6f}")
print()

# ---- 2) Classification report per classe
target_names = [label_names[i] for i in sorted(label_names.keys())]

print("=== CLASSIFICATION REPORT ===")
print(classification_report(
    y_val,
    y_val_pred,
    labels=sorted(label_names.keys()),
    target_names=target_names,
    digits=4
))

# ---- 3) Confusion matrix come DataFrame leggibile
cm = confusion_matrix(y_val, y_val_pred, labels=sorted(label_names.keys()))
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)

print("=== CONFUSION MATRIX (counts) ===")
display(cm_df)

# ---- 4) Confusion matrix normalizzata per riga (recall per classe)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
cm_norm_df = pd.DataFrame(cm_norm, index=target_names, columns=target_names)

print("=== CONFUSION MATRIX (row-normalized: recall profile) ===")
display(cm_norm_df.round(3))

# ---- 5) Top confusion pairs (dove sbaglia di più)
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)

pairs = []
for i, true_name in enumerate(target_names):
    for j, pred_name in enumerate(target_names):
        if i != j and off_diag[i, j] > 0:
            pairs.append((off_diag[i, j], true_name, pred_name))

pairs.sort(reverse=True, key=lambda x: x[0])

print("=== TOP 15 CONFUSIONS (True -> Pred) ===")
for k, (cnt, t, p) in enumerate(pairs[:15], start=1):
    print(f"{k:02d}. {t} -> {p}: {cnt}")


=== GLOBAL METRICS ===
Macro F1    : 0.684425
Micro F1    : 0.686000
Weighted F1 : 0.684980

=== CLASSIFICATION REPORT ===
                    precision    recall  f1-score   support

International News     0.7628    0.6549    0.7048      4709
          Business     0.7218    0.7951    0.7567      2118
        Technology     0.8017    0.7899    0.7958      2232
     Entertainment     0.5337    0.5353    0.5345      1995
            Sports     0.7816    0.9015    0.8373      1715
      General News     0.5260    0.5182    0.5221      2611
            Health     0.5469    0.7710    0.6399       620

          accuracy                         0.6860     16000
         macro avg     0.6678    0.7094    0.6844     16000
      weighted avg     0.6893    0.6860    0.6850     16000

=== CONFUSION MATRIX (counts) ===


,International News,Business,Technology,Entertainment,Sports,General News,Health
International News,3084,186,113,325,92,782,127
Business,85,1684,115,90,12,90,42
Technology,90,155,1763,117,16,39,52
Entertainment,206,126,125,1068,164,234,72
Sports,23,9,3,86,1546,46,2
General News,514,154,66,277,146,1353,101
Health,41,19,14,38,2,28,478


=== CONFUSION MATRIX (row-normalized: recall profile) ===


,International News,Business,Technology,Entertainment,Sports,General News,Health
International News,0.655,0.039,0.024,0.069,0.020,0.166,0.027
Business,0.040,0.795,0.054,0.042,0.006,0.042,0.020
Technology,0.040,0.069,0.790,0.052,0.007,0.017,0.023
Entertainment,0.103,0.063,0.063,0.535,0.082,0.117,0.036
Sports,0.013,0.005,0.002,0.050,0.901,0.027,0.001
General News,0.197,0.059,0.025,0.106,0.056,0.518,0.039
Health,0.066,0.031,0.023,0.061,0.003,0.045,0.771


=== TOP 15 CONFUSIONS (True -> Pred) ===
01. International News -> General News: 782
02. General News -> International News: 514
03. International News -> Entertainment: 325
04. General News -> Entertainment: 277
05. Entertainment -> General News: 234
06. Entertainment -> International News: 206
07. International News -> Business: 186
08. Entertainment -> Sports: 164
09. Technology -> Business: 155
10. General News -> Business: 154
11. General News -> Sports: 146
12. International News -> Health: 127
13. Entertainment -> Business: 126
14. Entertainment -> Technology: 125
15. Technology -> Entertainment: 117
